# Total Training Notebook

이 notebook는 현재 정리된 `modeling_module` public API 기준으로 `total training`을 다시 실행하기 위한 런닝 문서입니다.

## 이 notebook에서 하는 일
- `tb_master_target.parquet` 기반으로 endogenous-only 전체 family 학습
- `tb_master_target_exo.parquet` 우선, 없으면 `tb_master_exo.parquet` + `tb_master_target.parquet` join 기반으로 endogenous + exogenous 전체 family 학습
- 각 학습 결과의 `manifest_path`, `ckpt_paths`, `save_dir` 확인

## 학습 대상 family
- `patchtst` -> `patchtst_base`, `patchtst_quantile`
- `patchmixer` -> `patchmixer_base`, `patchmixer_quantile`
- `titan` -> `titan_base`, `titan_lmm`, `titan_seq2seq`

즉 family 기준으로는 3개지만, 최종 supervised checkpoint는 기본적으로 7개가 생성됩니다.

## 이번 notebook의 기본 방향
- 기본값은 `weekly` full-train 기준의 **균형형(balanced)** 설정입니다.
- 목표는 `lookback=52` / 작은 patch 토큰 수 때문에 GPU가 너무 한가한 상태를 완화하는 것입니다.
- 그래서 기본 추천값을 다음처럼 올려 두었습니다.
  - `LOOKBACK = 104`
  - `patchtst.patch_len = 13`, `patchtst.stride = 6`, `d_model = 384`
  - `patchmixer.patch_len = 13`, `patchmixer.stride = 6`, `d_model = 256`
  - `titan.d_model = 384`, `n_layers = 4`, `n_heads = 8`
- 이 설정은 GPU 연산량을 늘리면서도, 현재 데이터 기준으로 `lookback=104`, `horizon=27`에서 여전히 약 79.5% ID가 학습 가능하다는 점을 고려한 절충안입니다.

## 먼저 볼 설정 포인트
- `LOOKBACK`, `HORIZON`, `WARMUP_EPOCHS`, `SPIKE_EPOCHS`, `SSL_PRETRAIN_EPOCHS`
- `ENDO_BATCH_SIZE`, `EXO_BATCH_SIZE`
- `MODEL_ARCHITECTURE`
- `USE_ID_SAMPLE`: 빠른 dry-run 용 샘플 학습 여부
- `TARGET_EXO_SOURCE`: 있으면 우선 사용, 없으면 자동 fallback + join
- `PAST_EXO_CONT_COLS`, `FUTURE_EXO_CONT_COLS`
- `NUM_WORKERS`, `PERSISTENT_WORKERS`, `PREFETCH_FACTOR`: GPU feeding을 위한 dataloader runtime 설정


In [1]:
from __future__ import annotations

import importlib.util
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import polars as pl
import torch


# Remote kernel이면 여기에 서버 기준 repo 절대경로를 넣을 수 있습니다.
# Example: REPO_ROOT_OVERRIDE = Path("/home/ubuntu/ts_forecaster_lib")
# repo clone이 서버에 없고 modeling_module만 설치되어 있어도 notebook는 동작하도록 구성합니다.
REPO_ROOT_OVERRIDE = None


def _looks_like_repo(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src").exists()


def _iter_named_repo_candidates(root: Path, repo_name: str = "ts_forecaster_lib", max_depth: int = 4):
    if not root.exists() or not root.is_dir():
        return

    try:
        root_resolved = root.resolve()
    except Exception:
        root_resolved = root

    stack = [(root_resolved, 0)]
    while stack:
        current, depth = stack.pop()
        if current.name == repo_name:
            yield current
        if depth >= max_depth:
            continue
        try:
            children = list(current.iterdir())
        except Exception:
            continue
        for child in children:
            if child.is_dir() and not child.name.startswith('.'):
                stack.append((child, depth + 1))


def find_repo_root(start: Path, explicit_repo_root: Path | None = None) -> Path | None:
    env_repo_root = os.environ.get("TS_FORECASTER_REPO_ROOT")

    candidates = []
    if explicit_repo_root is not None:
        candidates.append(Path(explicit_repo_root).expanduser().resolve())
    if env_repo_root:
        candidates.append(Path(env_repo_root).expanduser().resolve())
    candidates.extend([start, *start.parents])

    home = Path.home()
    common_roots = [
        home,
        home / "workspace",
        home / "workspaces",
        home / "projects",
        home / "PycharmProjects",
        Path("/workspace"),
        Path("/workspaces"),
        Path("/home"),
        Path("/root"),
    ]
    for root in common_roots:
        candidates.extend(_iter_named_repo_candidates(root))

    seen = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if _looks_like_repo(candidate):
            return candidate

    return None


def resolve_import_paths() -> tuple[Path | None, Path | None]:
    repo_root = find_repo_root(Path.cwd().resolve(), explicit_repo_root=REPO_ROOT_OVERRIDE)
    src_root = repo_root / "src" if repo_root is not None else None

    if src_root is not None and str(src_root) not in sys.path:
        sys.path.insert(0, str(src_root))

    spec = importlib.util.find_spec("modeling_module")
    if spec is None or spec.origin is None:
        raise RuntimeError(
            "Could not import modeling_module. Either set REPO_ROOT_OVERRIDE to the server repo path, "
            "set TS_FORECASTER_REPO_ROOT, or install the package on the remote environment."
        )

    module_init = Path(spec.origin).resolve()
    module_root = module_init.parent

    if repo_root is None and module_root.parent.name == "src":
        repo_root = module_root.parent.parent
        src_root = repo_root / "src"

    return repo_root, src_root


NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT, SRC_ROOT = resolve_import_paths()

from modeling_module import (
    ArchitectureConfig,
    ArtifactConfig,
    DataColumnConfig,
    DataRequest,
    DataWindowConfig,
    ExoTSTArchitectureConfig,
    ExogenousConfig,
    LoaderConfig,
    PatchMixerArchitectureConfig,
    PatchTSTArchitectureConfig,
    RuntimeConfig,
    SSLConfig,
    TitanArchitectureConfig,
    TrainRequest,
    TrainerConfig,
    build_dataloader,
    build_dataset,
    load_predictor,
    train,
)
from modeling_module.utils.device import select_default_device


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

DEFAULT_DEVICE, DEFAULT_DEVICE_DIAGNOSTIC = select_default_device()

print("REPO_ROOT:", REPO_ROOT)
print("SRC_ROOT :", SRC_ROOT)
print("PYTHON   :", sys.executable)
print("TORCH    :", torch.__version__)
print("DEVICE   :", DEFAULT_DEVICE)
if DEFAULT_DEVICE_DIAGNOSTIC:
    print("DEVICE_NOTE:", DEFAULT_DEVICE_DIAGNOSTIC)
print("TF32 matmul:", getattr(torch.backends.cuda.matmul, "allow_tf32", None) if torch.cuda.is_available() else None)
print("cuDNN benchmark:", torch.backends.cudnn.benchmark if hasattr(torch.backends, "cudnn") else None)


REPO_ROOT: /home/leekwanhyeong/workspace/ts_forecaster_lib
SRC_ROOT : /home/leekwanhyeong/workspace/ts_forecaster_lib/src
PYTHON   : /home/leekwanhyeong/miniconda3/envs/ai_env/bin/python
TORCH    : 2.11.0+cu130
DEVICE   : cuda
TF32 matmul: True
cuDNN benchmark: True


## 1. Run Config

여기서 학습 조건을 한 번에 바꿉니다.

- `USE_ID_SAMPLE=True` 이면 충분한 history가 있는 일부 id만 잘라서 빠르게 dry-run 할 수 있습니다.
- `TARGET_EXO_SOURCE` 가 있으면 그대로 쓰고, 없으면 `tb_master_exo.parquet` 와 `tb_master_target.parquet` 를 join해서 one-table exogenous 학습용 입력을 만듭니다.
- 현재 기본값은 **weekly full-train용 balanced 추천 세팅**입니다.
- `architecture`는 family 단위 override로 들어갑니다.
  - `patchtst` 설정은 `patchtst_base`, `patchtst_quantile` 모두에 적용
  - `patchmixer` 설정은 `patchmixer_base`, `patchmixer_quantile` 모두에 적용
  - `titan` 설정은 `titan_base`, `titan_lmm`, `titan_seq2seq` 모두에 적용


In [2]:
DATA_ROOT = REPO_ROOT / "raw_data" / "master"

TARGET_SOURCE = DATA_ROOT / "tb_master_target.parquet"
TARGET_EXO_SOURCE = DATA_ROOT / "tb_master_target_exo.parquet"  # 있으면 우선 사용
EXO_SOURCE_FALLBACK = DATA_ROOT / "tb_master_exo.parquet"

FREQ = "weekly"
ID_COL = "oper_part_no"
DATE_COL = "demand_dt"
Y_COL = "demand_qty"

# Balanced weekly recommendation:
# - lookback=104 keeps data coverage relatively healthy while increasing sequence compute.
# - larger family-level model overrides help the GPU do more useful work.
LOOKBACK = 104
HORIZON = 27
ENDO_BATCH_SIZE = 1024
EXO_BATCH_SIZE = 512
WARMUP_EPOCHS = 3
SPIKE_EPOCHS = 2
TRAIN_LR = 1e-3

# SSL은 현재 total train에서 PatchTST family에 실질적으로 적용됩니다.
SSL_MODE = "full"
SSL_PRETRAIN_EPOCHS = 2
SSL_MASK_RATIO = 0.3
SSL_LOSS_TYPE = "mse"
TRAIN_DEVICE = DEFAULT_DEVICE

NUM_WORKERS = 8
PIN_MEMORY = True
PERSISTENT_WORKERS = True
PREFETCH_FACTOR = 4
SHUFFLE = True

# 빠른 dry-run이 필요하면 True로 바꾸고 MAX_IDS를 줄여서 확인합니다.
USE_ID_SAMPLE = False
MAX_IDS = 256
MIN_OBSERVED_TARGET_ROWS = LOOKBACK + HORIZON

ENDO_FAMILY_MODELS = ["patchtst", "patchmixer", "titan"]
EXO_FAMILY_MODELS = ["patchtst", "patchmixer", "titan", "exotst"]

MODEL_ARCHITECTURE = ArchitectureConfig(
    patchtst=PatchTSTArchitectureConfig(
        patch_len=13,
        stride=6,
        d_model=384,
        n_layers=5,
        d_ff=1536,
        dropout=0.1,
        norm="LayerNorm",
        pre_norm=True,
        act="gelu",
        use_revin=True,
        pe="sincos",
        learn_pe=True,
        padding_patch="end",
    ),
    patchmixer=PatchMixerArchitectureConfig(
        patch_len=13,
        stride=6,
        d_model=192,
        e_layers=6,
        f_out=256,
        head_hidden=256,
        dropout=0.1,
        head_dropout=0.02,
        use_revin=True,
        final_nonneg=True,
        expander_n_harmonics=24,
    ),
    titan=TitanArchitectureConfig(
        d_model=384,
        n_layers=4,
        n_heads=8,
        d_ff=1536,
        dropout=0.1,
        contextual_mem_size=384,
        persistent_mem_size=96,
        use_revin=True,
        final_clamp_nonneg=False,
    ),
    exotst=ExoTSTArchitectureConfig(
        d_model=128,
        n_heads=8,
        d_ff=256,
        dropout=0.1,
        attn_dropout=0.1,
        exo_enc_layers=2,
        fusion_layers=2,
        endo_dec_layers=2,
        exo_memory_mode="all",
        exo_nan_policy="zero+indicator",
        use_revin=True,
        subtract_last=True,
    ),
)

PAST_EXO_CONT_COLS = [
    "sin_annual",
    "cos_annual",
    "sin_semi",
    "cos_semi",
    "sin_quarter",
    "cos_quarter",
    "weather_index",
    "macro_index",
    "promo_strength",
    "part_len",
    "week_of_year",
]

FUTURE_EXO_CONT_COLS = [
    "sin_annual",
    "cos_annual",
    "sin_semi",
    "cos_semi",
    "sin_quarter",
    "cos_quarter",
    "weather_index",
    "macro_index",
    "promo_strength",
    "week_of_year",
    "promo_flag",
    "supply_outage_flag",
    "peak_season_flag",
    "is_year_start",
    "is_year_end",
    "is_q_start",
    "is_q_end",
]

PAST_EXO_CAT_COLS = []

ARTIFACT_ROOT = REPO_ROOT / "artifacts" / "total_train"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

print("TARGET_SOURCE           :", TARGET_SOURCE)
print("TARGET_EXO_SOURCE       :", TARGET_EXO_SOURCE, "(exists=", TARGET_EXO_SOURCE.exists(), ")")
print("EXO_SOURCE_FALLBACK     :", EXO_SOURCE_FALLBACK, "(exists=", EXO_SOURCE_FALLBACK.exists(), ")")
print("ENDO_FAMILY_MODELS      :", ENDO_FAMILY_MODELS)
print("EXO_FAMILY_MODELS       :", EXO_FAMILY_MODELS)
print("LOOKBACK                :", LOOKBACK)
print("HORIZON                 :", HORIZON)
print("MIN_OBSERVED_TARGET_ROWS:", MIN_OBSERVED_TARGET_ROWS)
print("ENDO_BATCH_SIZE         :", ENDO_BATCH_SIZE)
print("EXO_BATCH_SIZE          :", EXO_BATCH_SIZE)
print("WARMUP_EPOCHS           :", WARMUP_EPOCHS)
print("SPIKE_EPOCHS            :", SPIKE_EPOCHS)
print("SSL_MODE                :", SSL_MODE)
print("SSL_PRETRAIN_EPOCHS     :", SSL_PRETRAIN_EPOCHS)
print("NUM_WORKERS             :", NUM_WORKERS)
print("PERSISTENT_WORKERS      :", PERSISTENT_WORKERS)
print("PREFETCH_FACTOR         :", PREFETCH_FACTOR)
print("MODEL_ARCHITECTURE      :", MODEL_ARCHITECTURE)
print("ARTIFACT_ROOT           :", ARTIFACT_ROOT)


TARGET_SOURCE           : /home/leekwanhyeong/workspace/ts_forecaster_lib/raw_data/master/tb_master_target.parquet
TARGET_EXO_SOURCE       : /home/leekwanhyeong/workspace/ts_forecaster_lib/raw_data/master/tb_master_target_exo.parquet (exists= False )
EXO_SOURCE_FALLBACK     : /home/leekwanhyeong/workspace/ts_forecaster_lib/raw_data/master/tb_master_exo.parquet (exists= True )
TOTAL_FAMILY_MODELS     : ['patchtst', 'patchmixer', 'titan']
LOOKBACK                : 104
HORIZON                 : 27
MIN_OBSERVED_TARGET_ROWS: 131
ENDO_BATCH_SIZE         : 2048
EXO_BATCH_SIZE          : 1024
WARMUP_EPOCHS           : 3
SPIKE_EPOCHS            : 2
SSL_MODE                : full
SSL_PRETRAIN_EPOCHS     : 2
NUM_WORKERS             : 8
PERSISTENT_WORKERS      : True
PREFETCH_FACTOR         : 4
MODEL_ARCHITECTURE      : ArchitectureConfig(patchtst=PatchTSTArchitectureConfig(patch_len=13, stride=6, d_model=384, n_layers=5, d_ff=1536, dropout=0.1, norm='LayerNorm', pre_norm=True, act='gelu', use_rev

## 2. Helper Functions

아래 helper는 notebook를 실제 학습 런너처럼 쓰기 위한 공통 유틸입니다.

- parquet 로딩
- 충분한 history가 있는 id만 샘플링
- exogenous one-table 구성
- batch shape / training result summary 출력


In [3]:
def load_polars_table(source, table_name: str) -> pl.DataFrame:
    path = Path(source)
    if not path.exists():
        raise FileNotFoundError(f"{table_name} source not found: {path}")

    if path.suffix.lower() == ".parquet":
        return pl.read_parquet(path)
    if path.suffix.lower() in {".csv", ".txt"}:
        return pl.read_csv(path)

    raise ValueError(f"Unsupported file type for {table_name}: {path}")


def assert_columns(df: pl.DataFrame, required: list[str], table_name: str) -> None:
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"{table_name} is missing required columns: {missing}")


def sample_ids_by_observed_target(
    df: pl.DataFrame,
    *,
    id_col: str,
    y_col: str,
    min_obs: int,
    max_ids: int,
) -> list[str]:
    counts = (
        df.group_by(id_col)
        .agg(pl.col(y_col).is_not_null().sum().alias("observed_target_rows"))
        .filter(pl.col("observed_target_rows") >= min_obs)
        .sort("observed_target_rows", descending=True)
        .head(max_ids)
    )

    ids = counts[id_col].cast(pl.String).to_list()
    if not ids:
        raise ValueError(
            "No ids have enough observed target history. "
            f"Need at least {min_obs} non-null `{y_col}` rows per id."
        )
    return ids


def prepare_target_df(
    raw_df: pl.DataFrame,
    *,
    id_col: str,
    date_col: str,
    y_col: str,
    use_id_sample: bool,
    max_ids: int,
    min_obs: int,
) -> pl.DataFrame:
    assert_columns(raw_df, [id_col, date_col, y_col], "target table")

    df = (
        raw_df.sort([id_col, date_col])
        .with_columns(
            [
                pl.col(date_col).cast(pl.Int64),
                pl.col(y_col).cast(pl.Float64),
            ]
        )
    )

    if use_id_sample:
        keep_ids = sample_ids_by_observed_target(
            df,
            id_col=id_col,
            y_col=y_col,
            min_obs=min_obs,
            max_ids=max_ids,
        )
        df = df.filter(pl.col(id_col).is_in(keep_ids))

    return df


def resolve_exo_source(preferred: Path, fallback: Path) -> Path:
    if preferred.exists():
        return preferred
    if fallback.exists():
        return fallback
    raise FileNotFoundError(
        f"Could not find exogenous source. Checked: {preferred} and {fallback}"
    )


def prepare_exo_one_table(
    *,
    target_df: pl.DataFrame,
    exo_df: pl.DataFrame,
    id_col: str,
    date_col: str,
    y_col: str,
    past_exo_cont_cols: list[str],
    future_exo_cont_cols: list[str],
    past_exo_cat_cols: list[str],
) -> pl.DataFrame:
    assert_columns(exo_df, [id_col, date_col], "exogenous table")

    valid_ids = target_df.select(id_col).unique()
    one_table = exo_df.join(valid_ids, on=id_col, how="inner")

    if y_col not in one_table.columns:
        target_y = target_df.select([id_col, date_col, y_col])
        one_table = one_table.join(target_y, on=[id_col, date_col], how="left")

    required = [id_col, date_col, y_col, *past_exo_cont_cols, *future_exo_cont_cols, *past_exo_cat_cols]
    assert_columns(one_table, required, "one-table exogenous input")

    float_cols = sorted(
        set([y_col, *past_exo_cont_cols, *future_exo_cont_cols]).intersection(one_table.columns)
    )

    one_table = one_table.with_columns(
        [pl.col(date_col).cast(pl.Int64)] + [pl.col(col).cast(pl.Float64) for col in float_cols]
    )

    return one_table.sort([id_col, date_col])


def describe_batch(batch, label: str) -> None:
    names = ["x", "y", "uid_list", "future_exo", "past_exo_cont", "past_exo_cat"]
    print(f"=== {label} batch ===")
    for name, item in zip(names, batch):
        if torch.is_tensor(item):
            print(f"{name:14s}: shape={tuple(item.shape)}, dtype={item.dtype}")
        elif isinstance(item, list):
            print(f"{name:14s}: list(len={len(item)})")
        else:
            print(f"{name:14s}: {type(item).__name__}")


def result_summary_frame(train_result) -> pl.DataFrame:
    rows = []
    for model_key, ckpt_path in sorted(train_result.ckpt_paths.items()):
        rows.append(
            {
                "model_key": model_key,
                "ckpt_path": ckpt_path,
                "pretrain_ckpt_path": train_result.pretrain_ckpt_paths.get(model_key),
            }
        )

    if not rows:
        return pl.DataFrame(
            schema={
                "model_key": pl.String,
                "ckpt_path": pl.String,
                "pretrain_ckpt_path": pl.String,
            }
        )

    return pl.DataFrame(rows)


def print_training_result(train_result, label: str) -> pl.DataFrame:
    print(f"[{label}] requested_models:", train_result.requested_models)
    print(f"[{label}] save_dir       :", train_result.save_dir)
    print(f"[{label}] manifest_path  :", train_result.manifest_path)
    print(f"[{label}] total_ckpts    :", len(train_result.ckpt_paths))
    return result_summary_frame(train_result)


## 3. Endogenous-only Total Training

이 섹션은 `tb_master_target.parquet`만으로 전체 family 학습을 돌립니다.

- 입력: `oper_part_no`, `demand_dt`, `demand_qty`
- 모델 family: `patchtst`, `patchmixer`, `titan`
- 예상 최종 supervised checkpoint 수: 7개

먼저 아래 셀에서 데이터와 batch shape를 확인하고, 그 다음 학습 셀을 실행하면 됩니다.


In [4]:
target_raw = load_polars_table(TARGET_SOURCE, "tb_master_target")
target_df = prepare_target_df(
    target_raw,
    id_col=ID_COL,
    date_col=DATE_COL,
    y_col=Y_COL,
    use_id_sample=USE_ID_SAMPLE,
    max_ids=MAX_IDS,
    min_obs=MIN_OBSERVED_TARGET_ROWS,
)

print("target_raw shape:", target_raw.shape)
print("target_df shape :", target_df.shape)
print("n_unique ids    :", target_df.select(ID_COL).n_unique())
target_df.head(10)


target_raw shape: (14328097, 4)
target_df shape : (14328097, 4)
n_unique ids    : 49674


oper_part_no,demand_dt,demand_qty,seq
str,i64,f64,u32
"""DS_A001""",202029,4.0,1
"""DS_A001""",202030,2.0,2
"""DS_A001""",202031,2.0,3
"""DS_A001""",202032,5.0,4
"""DS_A001""",202033,1.0,5
"""DS_A001""",202034,5.0,6
"""DS_A001""",202035,5.0,7
"""DS_A001""",202036,4.0,8
"""DS_A001""",202037,5.0,9


In [5]:
endo_data_req = DataRequest(
    df=target_df,
    window=DataWindowConfig(
        lookback=LOOKBACK,
        horizon=HORIZON,
        freq=FREQ,
    ),
    columns=DataColumnConfig(
        id_col=ID_COL,
        date_col=DATE_COL,
        y_col=Y_COL,
    ),
    loader=LoaderConfig(
        stage="train",
        batch_size=ENDO_BATCH_SIZE,
        shuffle=SHUFFLE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
        prefetch_factor=PREFETCH_FACTOR,
    ),
)

endo_loader = build_dataloader(endo_data_req)
endo_batch = next(iter(endo_loader))

describe_batch(endo_batch, "ENDO/train")


=== ENDO/train batch ===
x             : shape=(2048, 104, 1), dtype=torch.float32
y             : shape=(2048, 27), dtype=torch.float32
uid_list      : list(len=2048)
future_exo    : shape=(2048, 27, 0), dtype=torch.float32
past_exo_cont : shape=(2048, 104, 0), dtype=torch.float32
past_exo_cat  : shape=(2048, 104, 0), dtype=torch.int64


In [6]:
endo_save_dir = ARTIFACT_ROOT / "endo_only"
endo_save_dir.mkdir(parents=True, exist_ok=True)

endo_train_req = TrainRequest(
    data=endo_data_req,
    freq=FREQ,
    models=ENDO_FAMILY_MODELS,
    architecture=MODEL_ARCHITECTURE,
    trainer=TrainerConfig(
        warmup_epochs=WARMUP_EPOCHS,
        spike_epochs=SPIKE_EPOCHS,
        lr=TRAIN_LR,
    ),
    ssl=SSLConfig(
        mode=SSL_MODE,
        pretrain_epochs=SSL_PRETRAIN_EPOCHS,
        mask_ratio=SSL_MASK_RATIO,
        loss_type=SSL_LOSS_TYPE,
    ),
    runtime=RuntimeConfig(
        device=TRAIN_DEVICE,
    ),
    artifacts=ArtifactConfig(
        save_dir=str(endo_save_dir),
        auto_save_dir=False,
    ),
)

endo_result = train(endo_train_req)
print_training_result(endo_result, "ENDO")


[exo_policy] use_exo=False | future(has=True, dim=0) | past(cont=0, cat=0)
[total_train][EXO] use_exo=False source=none exo_dim=0 future_cb=False past_cont=0 past_cat=0

[total_train] === RUN: patchtst (weekly) targets=['patchtst_base', 'patchtst_quantile'] ===
[SSL] PatchTST Pretrain (Weekly) -> /home/leekwanhyeong/workspace/ts_forecaster_lib/artifacts/total_train/endo_only/pretrain/patchtst_pretrain_best.pt
[train_patchtst_pretrain] Effective TrainingConfig:
{
  "device": "cuda",
  "log_every": 100,
  "lookback": 104,
  "horizon": 27,
  "epochs": 0,
  "lr": 0.001,
  "weight_decay": 0.001,
  "t_max": 40,
  "patience": 100,
  "max_grad_norm": 30.0,
  "amp_device": "cuda",
  "use_amp": true,
  "loss_mode": "auto",
  "point_loss": "mse",
  "huber_delta": 0.8,
  "q_star": 0.5,
  "use_cost_q_star": false,
  "Cu": 2.0,
  "Co": 1.0,
  "quantiles": [
    0.1,
    0.5,
    0.9
  ],
  "loss": "MAE()",
  "out_mul": 1,
  "param_names": null,
  "dist_name": "normal",
  "dist_scale_transform": "sof

OutOfMemoryError: CUDA out of memory. Tried to allocate 918.00 MiB. GPU 0 has a total capacity of 15.46 GiB of which 868.31 MiB is free. Process 322157 has 286.00 MiB memory in use. Process 328071 has 304.00 MiB memory in use. Including non-PyTorch memory, this process has 13.88 GiB memory in use. Of the allocated memory 12.97 GiB is allocated by PyTorch, and 592.02 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## 4. Endogenous + Exogenous Total Training

이 섹션은 exogenous one-table 경로를 사용합니다.

### 소스 선택 규칙
1. `tb_master_target_exo.parquet` 가 있으면 그 파일을 우선 사용
2. 없으면 `tb_master_exo.parquet` 를 읽고 `tb_master_target.parquet` 의 `demand_qty` 를 join

### 현재 exogenous 해석
- `PAST_EXO_CONT_COLS`: lookback 구간에 들어갈 과거 연속형 외생변수
- `FUTURE_EXO_CONT_COLS`: horizon 구간에 들어갈 known future covariate
- `PAST_EXO_CAT_COLS`: 현재 기본값은 빈 리스트

역시 먼저 batch shape를 확인한 뒤, 그 다음 학습 셀을 실행하면 됩니다.


In [ ]:
target_raw_for_exo = load_polars_table(TARGET_SOURCE, "tb_master_target")
target_df_for_exo = prepare_target_df(
    target_raw_for_exo,
    id_col=ID_COL,
    date_col=DATE_COL,
    y_col=Y_COL,
    use_id_sample=USE_ID_SAMPLE,
    max_ids=MAX_IDS,
    min_obs=MIN_OBSERVED_TARGET_ROWS,
)

exo_source = resolve_exo_source(TARGET_EXO_SOURCE, EXO_SOURCE_FALLBACK)
exo_raw = load_polars_table(exo_source, exo_source.name)
exo_one_table = prepare_exo_one_table(
    target_df=target_df_for_exo,
    exo_df=exo_raw,
    id_col=ID_COL,
    date_col=DATE_COL,
    y_col=Y_COL,
    past_exo_cont_cols=PAST_EXO_CONT_COLS,
    future_exo_cont_cols=FUTURE_EXO_CONT_COLS,
    past_exo_cat_cols=PAST_EXO_CAT_COLS,
)

print("exo_source        :", exo_source)
print("target_df_for_exo :", target_df_for_exo.shape)
print("exo_raw shape     :", exo_raw.shape)
print("exo_one_table     :", exo_one_table.shape)
print("n_unique ids      :", exo_one_table.select(ID_COL).n_unique())
exo_one_table.head(10)


In [ ]:
exo_data_req = DataRequest(
    df=exo_one_table,
    window=DataWindowConfig(
        lookback=LOOKBACK,
        horizon=HORIZON,
        freq=FREQ,
    ),
    columns=DataColumnConfig(
        id_col=ID_COL,
        date_col=DATE_COL,
        y_col=Y_COL,
    ),
    exogenous=ExogenousConfig(
        use_exogenous_mode=True,
        past_exo_cont_cols=PAST_EXO_CONT_COLS,
        past_exo_cat_cols=PAST_EXO_CAT_COLS,
        future_exo_cont_cols=FUTURE_EXO_CONT_COLS,
    ),
    loader=LoaderConfig(
        stage="train",
        batch_size=EXO_BATCH_SIZE,
        shuffle=SHUFFLE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
        prefetch_factor=PREFETCH_FACTOR,
    ),
)

exo_loader = build_dataloader(exo_data_req)
exo_batch = next(iter(exo_loader))

describe_batch(exo_batch, "EXO/train")


In [ ]:
exo_save_dir = ARTIFACT_ROOT / "endo_plus_exo"
exo_save_dir.mkdir(parents=True, exist_ok=True)

exo_train_req = TrainRequest(
    data=exo_data_req,
    models=EXO_FAMILY_MODELS,
    architecture=MODEL_ARCHITECTURE,
    trainer=TrainerConfig(
        warmup_epochs=WARMUP_EPOCHS,
        spike_epochs=SPIKE_EPOCHS,
        lr=TRAIN_LR,
    ),
    ssl=SSLConfig(
        mode=SSL_MODE,
        pretrain_epochs=SSL_PRETRAIN_EPOCHS,
        mask_ratio=SSL_MASK_RATIO,
        loss_type=SSL_LOSS_TYPE,
    ),
    runtime=RuntimeConfig(
        device=TRAIN_DEVICE,
    ),
    artifacts=ArtifactConfig(
        save_dir=str(exo_save_dir),
        auto_save_dir=False,
    ),
)

exo_result = train(exo_train_req)
print_training_result(exo_result, "EXO")


## 5. Notes

- 현재 기본 설정은 `lookback=104`, `warmup=3`, `spike=2`, `ssl_pretrain=2`, `ssl mode=full` 입니다.
- batch는 `endo=2048`, `exo=1024` 로 분리해 두었습니다. exogenous one-table 경로가 더 무거워서 기본값을 보수적으로 잡았습니다.
- `MODEL_ARCHITECTURE` 는 family 단위 override입니다.
  - `patchtst` 설정은 `patchtst_base`, `patchtst_quantile` 모두에 적용됩니다.
  - `patchmixer` 설정은 `patchmixer_base`, `patchmixer_quantile` 모두에 적용됩니다.
  - `titan` 설정은 `titan_base`, `titan_lmm`, `titan_seq2seq` 모두에 적용됩니다.
- `ssl mode=full` 은 현재 실질적으로 PatchTST family에만 SSL pretrain을 적용하고, PatchMixer/Titan은 supervised stage만 수행합니다.
- 현재 notebook는 `num_workers=8`, `persistent_workers=True`, `prefetch_factor=4` 를 설정합니다. `train()` 경로도 이 runtime loader 설정을 실제로 반영합니다.
- bootstrap 셀에서 `TF32`, `cuDNN benchmark` 를 켜 두었습니다. Ampere/Ada 계열 GPU에서는 보통 이 편이 더 낫습니다.
- `USE_ID_SAMPLE=True` 로 바꾸면 충분한 history가 있는 일부 id만 학습에 사용해 빠르게 점검할 수 있습니다.
- artifact는 기본적으로 `artifacts/total_train/endo_only`, `artifacts/total_train/endo_plus_exo` 아래에 저장됩니다.
- exogenous 섹션은 `tb_master_target_exo.parquet` 가 생기면 자동으로 그 파일을 사용하고, 없으면 현재처럼 `tb_master_exo.parquet` + target join 경로를 탑니다.
